In [ ]:
from IPython.display import display, HTML
display(HTML('<style>.container{width:100% !important;}</style>'))

In [ ]:
# -*- coding: utf-8 -*-

import json
from copy import copy
from queue import LifoQueue
import matplotlib.pyplot as plt

fontdict_text = {"family": "FangSong",
                 "style": "normal", "variant": "small-caps",
                 "weight": "black", "size": 10}

fontdict_title = {"family": "FangSong",
                  "style": "normal", "variant": "small-caps",
                  "weight": "black", "size": 15}

colors = {'blue1': '#E7F2FC',
          'blue2': '#CBE3F8',
          'blue3': '#AAD3F5',
          'blue4': '#198CD9',
          'orange1': '#FFF4E8',
          'orange2': '#FFE8CD',
          'orange3': '#FFDCAC',
          'orange4': '#F4996B',
          'gray': '#EBEBEB',
          'black': '#000000',
          }

In [ ]:
def format_name(name, ch, num=20):
    strs = name.split(ch)
    res = ""
    count = 0
    lines = 2
    for str in strs:
        if count + len(str) < num:
            res += str
            res += ch
            count += len(res)
        else:
            res += "\n"
            res += str
            res += ch
            count = 0
            lines += 1
    if res[-1] == ch:
        res = res[:-1]
    if res[-1] == "\n":
        res = res[:-1]
        lines -= 1
    return [res, lines]

def read_json(file):
    fd = open(file, 'r')
    content = fd.read()
    topics = json.loads(content)
    fd.close()
    return topics

topics = read_json('topic.json')

In [ ]:
component_list = []

class Border:
    def __init__(self, name, border, color):
        self.name = name
        self.border = border
        self.color = color

    def update(self, block):
        if self.border[0] < block.xy[1] + block.height:
            self.border[0] = block.xy[1] + block.height
        if self.border[1] > block.xy[0]:
            self.border[1] = block.xy[0]
        if self.border[2] > block.xy[1]:
            self.border[2] = block.xy[1]
        if self.border[3] < block.xy[0] + block.width:
            self.border[3] = block.xy[0] + block.width

    def expand(self):
        self.border[0] += 45
        self.border[1] -= 25
        self.border[2] -= 25
        self.border[3] += 25

    def merge(src1, src2):
        dst = copy(src1)
        dst[0] = max(src1[0], src2[0])
        dst[1] = min(src1[1], src2[1])
        dst[2] = min(src1[2], src2[2])
        dst[3] = max(src1[3], src2[3])
        return dst

    def draw(self):
        drawer = Block(self.name, [self.border[1], self.border[2]], [self.border[3] -
                       self.border[1], self.border[0] - self.border[2]], self.color)
        drawer.draw(outdegree=False, is_title=True)


class Block:
    def __init__(self, name, xy, wh, color, parent=''):
        self.name = name
        self.xy = xy
        self.width = wh[0]
        self.height = wh[1]
        self.color = color
        self.parent = parent

    def draw_name(self, fontdict=fontdict_title):
        x = self.xy[0] + self.width / 2
        y = self.xy[1] + self.height / 2
        plt.text(x, y, self.name, ha='center', va='center', fontdict=fontdict)

    def draw_title(self, fontdict=fontdict_title):
        x = self.xy[0] + self.width / 2
        y = self.xy[1] + self.height - 25
        plt.text(x, y, self.name, ha='center', va='center', fontdict=fontdict)

    def draw_rect(self):
        rect = plt.Rectangle(xy=self.xy,
                             width=self.width,
                             height=self.height,
                             color=self.color,
                             fill=True)
        plt.gca().add_patch(rect)

    def draw_edge(self):
        plt.hlines(self.xy[1],             self.xy[0],
                   self.xy[0]+self.width,  color="black")
        plt.hlines(self.xy[1]+self.height, self.xy[0],
                   self.xy[0]+self.width,  color="black")
        plt.vlines(self.xy[0],             self.xy[1],
                   self.xy[1]+self.height, color="black")
        plt.vlines(self.xy[0]+self.width,  self.xy[1],
                   self.xy[1]+self.height, color="black")

    def draw_outdegree(self):
        x1 = self.xy[0] + 5
        x2 = self.xy[0] + self.width - 5
        y = self.xy[1] + self.height / 2
        plt.plot(x1, y, marker='o', color='black')
        plt.plot(x2, y, marker='o', color='black')

    def draw(self, outdegree=True, is_title=False, fontdict=fontdict_title):
        stack.put([self, outdegree, is_title, fontdict])

    def pop(self, outdegree=True, is_title=False, fontdict=fontdict_title):
        if is_title:
            self.draw_title()
        else:
            self.draw_name(fontdict)
        self.draw_rect()
        self.draw_edge()
        if outdegree:
            self.draw_outdegree()


class Component:
    def __init__(self, component, xy, width, height, color):
        self.xy = xy
        self.width = width
        self.height = height
        self.color = color
        self.hline = 20
        self.
        self.name = component['component_name']
        self.readers = component['reader_topics']
        self.writers = component['writer_topics']
        component_list.append(self)

    def draw_name(self):
        [self.name, lines] = format_name(self.name, '_')
        self.xy[1] -= self.hline * lines
        xy = copy(self.xy)
        name = Block(self.name, xy, [self.width,
                     self.hline * lines], self.color)
        name.draw(outdegree=False,fontdict=fontdict_title)

    def draw_writers(self):
        for writer in self.writers:
            [writer, lines] = format_name(writer, '/', 40)
            self.xy[1] -= self.hline * lines
            xy = copy(self.xy)
            topic = Block(writer, xy, [self.width,
                          self.hline * lines], self.color, self.name)
            topic.draw(outdegree=True, fontdict=fontdict_text)

    def draw(self):
        self.draw_name()
        self.draw_writers()


class Chunk:
    def __init__(self, column, xy, wh, index=[0, 0]):
        self.column = column
        self.index = index
        self.xy = xy
        self.width = wh[0]
        self.height = wh[1]
        self.wspace = 50
        self.hspace = 30
        self.hskip = [0] * column

    def append(self, component, border, color):
        mi = self.hskip.index(min(self.hskip))
        xy = copy(self.xy)
        xy[0] += (self.width + self.wspace) * mi
        xy[1] -= self.hskip[mi]
        comp = Component(component, xy, self.width, self.height, color)
        comp.draw()
        block = Block("", xy, [self.width, self.xy[1] - xy[1]], color)
        border.update(block)
        self.hskip[mi] = self.xy[1] - xy[1] + self.hspace

In [ ]:
base_xy = [0, 800]
base_wh = [120, 40]

def draw_base_blocks():
    groups = topics['groups']
    comp1 = groups[0]['components']
    comp2 = groups[1]['components']
    comp3 = groups[2]['components']
    comp4 = groups[3]['components']


    border1 = Border("ASW", [-1000, 1000, 1000, -1000], colors["blue2"])
    border2 = Border("BSW", [-1000, 1000, 1000, -1000], colors["blue2"])
    border3 = Border("ASW", [-1000, 1000, 1000, -1000], colors["orange2"])
    border4 = Border("BSW", [-1000, 1000, 1000, -1000], colors["orange2"])

    chunk1 = Chunk(3, [0, 800], base_wh)
    for component in comp1:
        chunk1.append(component, border1, colors["blue3"])
    border1.expand()
    border1.draw()

    border2.border[0] = border1.border[2] - 80
    border4.border[0] = border1.border[2] - 80
    border3.border[1] = border1.border[3] + 80
    border4.border[1] = border1.border[3] + 80

    chunk2 = Chunk(3, [0, border2.border[0]], base_wh)
    for component in comp2:
        chunk2.append(component, border2, colors["blue3"])
    border2.expand()
    border2.draw()

    chunk3 = Chunk(2, [border3.border[1], 800], base_wh)
    for component in comp3:
        chunk3.append(component, border3, colors["orange3"])
    border3.expand()
    border3.draw()

    chunk4 = Chunk(2, [border4.border[1], border4.border[0]], base_wh)
    for component in comp4:
        chunk4.append(component, border4, colors["orange3"])
    border4.expand()
    border4.draw()
    

def draw_other_blocks():
    vis = Block("可视化/数采", [-200, 700], base_wh, colors['gray'])
    vis.draw()

    cars = Block("车机", [-200, 500], base_wh, colors['gray'])
    cars.draw()

    lidar = Block("lidar", [-200, -250], base_wh, colors['gray'])
    lidar.draw()

    camera = Block("camera", [-200, -450], base_wh, colors['gray'])
    camera.draw()

    app = Block("仪表", [975, 750], base_wh, colors['gray'])
    app.draw()

    uss = Block("uss", [975, 300], base_wh, colors['gray'])
    uss.draw()

    radar = Block("radar", [975, 150], base_wh, colors['gray'])
    radar.draw()

    gnss = Block("gnss", [975, -50], base_wh, colors['gray'])
    gnss.draw()

    imu = Block("imu", [975, -200], base_wh, colors['gray'])
    imu.draw()

    can = Block("can", [975, -450], base_wh, colors['gray'])
    can.draw()

def draw_components():
    draw_base_blocks()
    draw_other_blocks()

In [ ]:
def get_component(name):
    groups = topics['groups']
    for group in groups:
        color = colors['blue4']
        if group['group_name'][:3] == 'mcu':
            color = colors['orange4']
        components = group['components']
        for component in components:
            if name == component['component_name']:
                return [component, color]

            
def draw_highlight(name):
    outdegrees = []
    [component, color] = get_component(name)
    while not stack.empty():
        [block, outdegree, is_title, fontdict] = stack.get()
        block.pop(outdegree, is_title, fontdict)
        [comp_name, _] = format_name(name, '_')
        if block.name == comp_name:
            block.color = color
            block.pop(outdegree, is_title, fontdict)
        for writer in component['writer_topics']:
            [writer_name, _] = format_name(writer, '/', 40)
            if comp_name == block.parent and block.name == writer_name:
                block.color = color
                block.pop(outdegree, is_title, fontdict)
                outdegrees.append(block)
    return outdegrees


def get_indegree(name, outdegree):
    [component, _] = get_component(name)
#     for writer in component['writer_topics']:
    for group in topics['groups']:
        comps = group['components']
        for comp in comps:
            for reader in comp['reader_topics']:
                [reader, _] = format_name(reader, '/', 40)
                if reader == outdegree.name:
                    start_xy = outdegree.xy
                    end_xy = get_pos(comp['component_name'])
                    plt.arrow(start_xy[0],start_xy[1],end_xy[0]-start_xy[0],end_xy[1]-start_xy[1],length_includes_head = True,head_width = 5,head_length = 10,fc = 'r',ec = 'r')
                    print(start_xy, end_xy)
                    indegrees.append(comp['component_name'])
                    break
    return indegrees


def get_pos(indegree):
    [indegree, _] = format_name(indegree, '_')
    for component in component_list:
        if component.name == indegree:
            return [component.xy[0], component.xy[1] + component.height]

def draw_outdegree(indegrees):
    for indegree in indegrees:
        pos = get_pos(indegree)

In [ ]:
stack = LifoQueue()
plt.figure(figsize=(32, 18))
draw_components()
indegrees = []
outdegrees = draw_highlight("panorama_view_camera_perception")

plt.show()